<a href="https://colab.research.google.com/github/MajidFQ/TaxoSplitter_VerticalRouter_v1/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MajidFQ/TaxoSplitter_VerticalRouter_v1.git


Cloning into 'TaxoSplitter_VerticalRouter_v1'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 55 (delta 24), reused 44 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 89.33 KiB | 9.93 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/TaxoSplitter_VerticalRouter_v1


In [2]:
%cd /content/TaxoSplitter_VerticalRouter_v1


/content/TaxoSplitter_VerticalRouter_v1


In [3]:
# ═══════════════════════════════════════════════════════════════
# COLAB SETUP
# ═══════════════════════════════════════════════════════════════

# 1. Install dependencies
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "torch", "datasets", "scikit-learn", "matplotlib"], check=True)

# 2. Verify GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Training will be slow. Go to Runtime > Change runtime type > T4 GPU")

# 3. Create all required directories
import os
os.makedirs("outputs", exist_ok=True)
os.makedirs("model", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("eval_results", exist_ok=True)
print("Directory structure ready.")

CUDA available: True
GPU: Tesla T4
Directory structure ready.


In [4]:
import os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt
from collections import defaultdict

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
import torch
from sklearn.manifold import TSNE

# ── Import data from the data/ package ──────────────────────────────────────
from data.categories import CATEGORIES
from data.domains import DOMAINS
from data.manual_corrections import MANUAL_CORRECTIONS

# ── Create all required output directories ─────────────────────────────────

print(f'Loaded {len(CATEGORIES)} categories')
print(f'Loaded {len(DOMAINS)} domains')
print(f'Loaded {len(MANUAL_CORRECTIONS)} manual corrections')


/tmp/ipykernel_1425/2210451378.py:9: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/tmp/ipykernel_1425/2210451378.py:10: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


Loaded 2023 categories
Loaded 15 domains
Loaded 62 manual corrections


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: ZERO-SHOT BASELINE
# Before any training, see how well the base model already does.
# This gives you a benchmark to know if training actually helped.
# ─────────────────────────────────────────────────────────────────────────────

def run_zero_shot(categories, domains, model_name="intfloat/e5-base-v2"):
    """
    Embed all domain descriptions and all categories.
    Assign each category to its nearest domain by cosine similarity.
    No training. Pure semantic matching.

    Why e5-base-v2?
    - Designed for asymmetric retrieval (long description vs short query)
    - Better than generic sentence-bert for this exact use case
    - Free, runs locally
    """
    print("Loading model...")
    model = SentenceTransformer(model_name)

    # e5 models need a prefix to distinguish query vs passage
    # Domain descriptions = passages (what we're searching through)
    # Categories = queries (what we're searching with)
    domain_names = list(domains.keys())
    domain_texts = [f"passage: {desc.strip()}" for desc in domains.values()]
    category_texts = [f"query: {cat}" for cat in categories]

    print("Embedding domains...")
    domain_embeddings = model.encode(domain_texts, normalize_embeddings=True, show_progress_bar=False)

    print("Embedding categories...")
    category_embeddings = model.encode(category_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=64)

    # Cosine similarity matrix: (num_categories x num_domains)
    similarity_matrix = category_embeddings @ domain_embeddings.T

    results = {}
    for i, category in enumerate(categories):
        scores = similarity_matrix[i]
        best_domain_idx = scores.argmax()
        best_domain = domain_names[best_domain_idx]
        best_score = float(scores[best_domain_idx])

        # Flag low confidence — these are your problem cases
        confidence = "HIGH" if best_score > 0.75 else "MEDIUM" if best_score > 0.60 else "LOW"

        results[category] = {
            "domain": best_domain,
            "score": round(best_score, 4),
            "confidence": confidence,
            "all_scores": {domain_names[j]: round(float(scores[j]), 4) for j in range(len(domain_names))}
        }

    return results

# Run it
baseline_results = run_zero_shot(CATEGORIES, DOMAINS)

# Print results grouped by domain
by_domain = defaultdict(list)
for cat, result in baseline_results.items():
    by_domain[result["domain"]].append((cat, result["score"], result["confidence"]))

print("\n" + "="*60)
print("ZERO-SHOT RESULTS")
print("="*60)
for domain, cats in sorted(by_domain.items()):
    print(f"\n{domain} ({len(cats)} categories):")
    for cat, score, conf in sorted(cats, key=lambda x: x[1], reverse=True):
        flag = "⚠️" if conf == "LOW" else "❓" if conf == "MEDIUM" else "✓"
        print(f"  {flag} {cat:<45} {score:.4f}")

# Save to outputs/ directory
with open("outputs/baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)
print("\nSaved to outputs/baseline_results.json")

Loading model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding domains...
Embedding categories...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


ZERO-SHOT RESULTS

ARCHIVE (142 categories):
  ✓ natural feature                               0.8093
  ✓ business park                                 0.8055
  ✓ landmark                                      0.7975
  ✓ historical landmark                           0.7871
  ✓ public utility company                        0.7862
  ✓ cultural landmark                             0.7855
  ✓ public services                               0.7851
  ✓ public beach                                  0.7832
  ✓ department of transportation                  0.7800
  ✓ tourist attraction                            0.7784
  ✓ exhibition and trade center                   0.7783
  ✓ public library                                0.7747
  ✓ state park                                    0.7746
  ✓ toll road rest stop                           0.7738
  ✓ oil & natural gas company                     0.7736
  ✓ bus ticket agency                             0.7730
  ✓ natural stone supplier                

In [6]:
# ─────────────────────────────────────────────────────────────
# STEP 1: LOAD GROUND TRUTH & CHECK DOMAINS
# ─────────────────────────────────────────────────────────────
import json
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

with open("data/ground_truth.json") as f:
    ground_truth = json.load(f)

from data.domains import DOMAINS

# Check for any mismatch between ground truth domains and domains.py
gt_domains = set([v for v in ground_truth.values() if v is not None])
my_domains = set(DOMAINS.keys())

print("In ground truth but NOT in domains.py:", gt_domains - my_domains)
print("In domains.py but NOT in ground truth:", my_domains - gt_domains)

if gt_domains - my_domains:
    print("\nWARNING: Mismatches found! Stop and fix domains.py to match ground_truth.json exactly.")
else:
    print("\nDomain keys match perfectly.")

print(f"\nGround truth categories loaded: {len(ground_truth)}")
print(f"Domains loaded: {len(DOMAINS)}")


In ground truth but NOT in domains.py: set()
In domains.py but NOT in ground truth: set()

Domain keys match perfectly.

Ground truth categories loaded: 2023
Domains loaded: 15


In [7]:
# ─────────────────────────────────────────────────────────────
# STEP 2: BUILD TRAINING EXAMPLES
# ─────────────────────────────────────────────────────────────
def build_training_examples(ground_truth, domains):
    examples = []
    skipped = 0
    for category, correct_domain in ground_truth.items():
        if correct_domain is None or correct_domain not in domains:
            skipped += 1
            continue

        correct_desc = domains[correct_domain].strip()

        # POSITIVE pair: category <-> correct domain (Label = 1.0)
        examples.append(InputExample(
            texts=[category, correct_desc],
            label=1.0
        ))

        # NEGATIVE pairs: category <-> all wrong domains (Label = 0.0)
        for domain_name, domain_desc in domains.items():
            if domain_name == correct_domain:
                continue
            examples.append(InputExample(
                texts=[category, domain_desc.strip()],
                label=0.0
            ))

    print(f"Training examples built: {len(examples)}")
    print(f"Skipped (missing/mismatch): {skipped}")
    return examples

training_examples = build_training_examples(ground_truth, DOMAINS)


Training examples built: 30345
Skipped (missing/mismatch): 0


In [8]:
# ─────────────────────────────────────────────────────────────
# STEP 3: TRAIN THE EMBEDDING MODEL
# ─────────────────────────────────────────────────────────────
def train(training_examples, output_path="model/"):
    print("Loading base model...")
    model = SentenceTransformer("intfloat/e5-base-v2")

    loader = DataLoader(
        training_examples,
        shuffle=True,
        batch_size=16  # Safe for Colab T4
    )

    loss = losses.CosineSimilarityLoss(model)

    print(f"\nTraining started...")
    print(f"Examples: {len(training_examples)}")
    print(f"Batches per epoch: {len(loader)}")
    print(f"Epochs: 4")
    print(f"Total steps: {len(loader) * 4}\n")

    model.fit(
        train_objectives=[(loader, loss)],
        epochs=4,
        warmup_steps=50,
        output_path=output_path,
        show_progress_bar=True,
        save_best_model=True,
        checkpoint_path="model/checkpoints/",
        checkpoint_save_steps=200,
    )

    print(f"\nDone. Model saved to: {output_path}")
    return model

trained_model = train(training_examples)


Loading base model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Training started...
Examples: 30345
Batches per epoch: 1897
Epochs: 4
Total steps: 7588



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.058766
1000,0.023635
1500,0.015943
2000,0.015759
2500,0.008347
3000,0.007082
3500,0.007202
4000,0.005027
4500,0.003701
5000,0.002426


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Done. Model saved to: model/


In [9]:
# ─────────────────────────────────────────────────────────────
# STEP 4: EVALUATE AGAINST GROUND TRUTH
# ─────────────────────────────────────────────────────────────
def evaluate(ground_truth, domains, model):
    domain_names = list(domains.keys())
    domain_embeddings = model.encode(
        [f"passage: {desc.strip()}" for desc in domains.values()],
        normalize_embeddings=True,
        show_progress_bar=False
    )

    correct = 0
    total = 0
    wrong_cases = []
    per_domain_correct = defaultdict(int)
    per_domain_total = defaultdict(int)

    valid_gt = {k: v for k, v in ground_truth.items() if v is not None}
    categories = list(valid_gt.keys())
    true_labels = list(valid_gt.values())

    cat_embeddings = model.encode(
        [f"query: {cat}" for cat in categories],
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True
    )

    sim_matrix = cat_embeddings @ domain_embeddings.T

    for i, category in enumerate(categories):
        true_domain = true_labels[i]
        if true_domain not in domain_names:
            continue

        predicted_domain = domain_names[sim_matrix[i].argmax()]
        score = float(sim_matrix[i].max())

        per_domain_total[true_domain] += 1
        total += 1

        if predicted_domain == true_domain:
            correct += 1
            per_domain_correct[true_domain] += 1
        else:
            wrong_cases.append({
                "category": category,
                "true": true_domain,
                "predicted": predicted_domain,
                "score": round(score, 4)
            })

    overall_acc = correct / total * 100
    print(f"\n{'='*60}")
    print(f"OVERALL ACCURACY: {correct}/{total} = {overall_acc:.1f}%")
    print(f"{'='*60}")

    print(f"\nPER-DOMAIN ACCURACY:")
    for domain in sorted(domain_names):
        if per_domain_total[domain] == 0:
            continue
        acc = per_domain_correct[domain] / per_domain_total[domain] * 100
        bar = "█" * int(acc / 5)
        print(f"  {domain:<25} {acc:5.1f}%  {bar}")

    print(f"\nWRONG PREDICTIONS ({len(wrong_cases)}):")
    for w in sorted(wrong_cases, key=lambda x: x["score"]):
        print(f"  {w['category']:<40} true={w['true']:<20} predicted={w['predicted']}")

    with open("eval_results/wrong_predictions.json", "w") as f:
        json.dump(wrong_cases, f, indent=2)

    return overall_acc, wrong_cases

accuracy, wrong = evaluate(ground_truth, DOMAINS, trained_model)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


OVERALL ACCURACY: 2012/2023 = 99.5%

PER-DOMAIN ACCURACY:
  ARCHIVE                    99.0%  ███████████████████
  D01_Healthcare            100.0%  ████████████████████
  D02_PetCare               100.0%  ████████████████████
  D03_Automotive            100.0%  ████████████████████
  D04_Construction          100.0%  ████████████████████
  D05_Cleaning              100.0%  ████████████████████
  D06_RealEstate            100.0%  ████████████████████
  D07_FoodDining            100.0%  ████████████████████
  D08_Lodging               100.0%  ████████████████████
  D09_BeautyWellness        100.0%  ████████████████████
  D10_FitnessSports         100.0%  ████████████████████
  D11_Retail                 99.7%  ███████████████████
  D12_B2BCorporate           97.7%  ███████████████████
  D13_Manufacturing          99.1%  ███████████████████
  D14_PublicNonProfit       100.0%  ████████████████████

WRONG PREDICTIONS (11):
  catering food and drink supplier         true=D12_B2BCorporate 

In [10]:
# ─────────────────────────────────────────────────────────────
# STEP 5: BUILD FINAL LOOKUP TABLE
# ─────────────────────────────────────────────────────────────
def build_lookup_table(ground_truth, domains, model):
    domain_names = list(domains.keys())
    domain_embeddings = model.encode(
        [f"passage: {desc.strip()}" for desc in domains.values()],
        normalize_embeddings=True
    )

    lookup = {}
    valid_gt = {k: v for k, v in ground_truth.items() if v is not None}
    categories = list(valid_gt.keys())

    embeddings = model.encode(
        [f"query: {c}" for c in categories],
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True
    )

    sim = embeddings @ domain_embeddings.T
    for i, cat in enumerate(categories):
        best_idx = sim[i].argmax()
        lookup[cat] = {
            "domain": domain_names[best_idx],
            "score": round(float(sim[i].max()), 4),
            "ground_truth": valid_gt[cat],
            "correct": domain_names[best_idx] == valid_gt[cat]
        }

    with open("outputs/final_mapping.json", "w") as f:
        json.dump(lookup, f, indent=2)

    print(f"Lookup table saved: outputs/final_mapping.json")
    return lookup

lookup = build_lookup_table(ground_truth, DOMAINS, trained_model)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Lookup table saved: outputs/final_mapping.json


# ════════════════════════════════════════════════════════════
# Figures for Research Paper
# ════════════════════════════════════════════════════════════
All figures saved at **300 DPI** to `figures/` directory.


In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np

def plot_confidence_distribution(baseline_results, final_lookup):
    baseline_scores = [r['score'] for r in baseline_results.values() if 'score' in r]
    finetuned_scores = [r['score'] for r in final_lookup.values()]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(baseline_scores, bins=50, alpha=0.6, color='#e74c3c', label='Baseline (zero-shot)', edgecolor='white', linewidth=0.5)
    ax.hist(finetuned_scores, bins=50, alpha=0.6, color='#2ecc71', label='Fine-tuned', edgecolor='white', linewidth=0.5)
    ax.axvline(x=0.75, color='#f39c12', linestyle='--', linewidth=2, label='HIGH threshold (0.75)')
    ax.axvline(x=0.60, color='#9b59b6', linestyle='--', linewidth=2, label='MEDIUM threshold (0.60)')
    ax.set_xlabel('Cosine Similarity Score', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Categories', fontsize=13, fontweight='bold')
    ax.set_title('Confidence Distribution: Baseline vs Fine-Tuned', fontsize=15, fontweight='bold', pad=15)
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xlim(0.3, 1.0)
    plt.tight_layout()
    plt.savefig('figures/confidence_dist.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: figures/confidence_dist.png')

def plot_per_domain_accuracy(final_lookup):
    domains_list = sorted(list(set([r['ground_truth'] for r in final_lookup.values()])))
    domain_total = defaultdict(int)
    domain_correct = defaultdict(int)

    for cat, res in final_lookup.items():
        gt = res['ground_truth']
        domain_total[gt] += 1
        if res['correct']:
            domain_correct[gt] += 1

    finetuned_acc = [100.0 * domain_correct[d] / domain_total[d] if domain_total[d] > 0 else 0 for d in domains_list]
    x = np.arange(len(domains_list))
    width = 0.6

    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.bar(x, finetuned_acc, width, color='#2ecc71', alpha=0.85, edgecolor='white')

    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    short_labels = [d.split('_', 1)[1] if '_' in d else d for d in domains_list]
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=35, ha='right', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
    ax.set_title('Per-Domain Classification Accuracy (Fine-Tuned)', fontsize=15, fontweight='bold', pad=15)
    ax.set_ylim(0, 115)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('figures/per_domain_accuracy.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: figures/per_domain_accuracy.png')

def plot_tsne(categories, domains, model, title, save_path):
    domain_names = list(domains.keys())
    domain_texts = [f'passage: {desc.strip()}' for desc in domains.values()]
    category_texts = [f'query: {cat}' for cat in categories]

    print(f'  Encoding {len(categories)} categories...')
    domain_embeddings = model.encode(domain_texts, normalize_embeddings=True, show_progress_bar=False)
    category_embeddings = model.encode(category_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

    sim_matrix = category_embeddings @ domain_embeddings.T
    assignments = [domain_names[row.argmax()] for row in sim_matrix]
    all_embeddings = np.vstack([category_embeddings, domain_embeddings])

    print('  Running t-SNE (this may take a minute)...')
    tsne = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000)
    coords = tsne.fit_transform(all_embeddings)

    cat_coords = coords[:len(categories)]
    dom_coords = coords[len(categories):]

    # Palette for 15 domains (14 + ARCHIVE)
    palette = [
        '#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6', '#1abc9c', '#e67e22',
        '#34495e', '#e91e63', '#00bcd4', '#8bc34a', '#795548', '#FF9800', '#673AB7', '#607D8B'
    ]
    domain_to_color = {name: palette[i % len(palette)] for i, name in enumerate(domain_names)}

    fig, ax = plt.subplots(figsize=(16, 12))
    for domain in domain_names:
        mask = [i for i, a in enumerate(assignments) if a == domain]
        if mask:
            short = domain.split('_', 1)[1] if '_' in domain else domain
            ax.scatter(cat_coords[mask, 0], cat_coords[mask, 1], c=domain_to_color[domain], s=12, alpha=0.5, label=f'{short} ({len(mask)})', edgecolors='none')

    for j, name in enumerate(domain_names):
        short = name.split('_', 1)[1] if '_' in name else name
        ax.scatter(dom_coords[j, 0], dom_coords[j, 1], marker='*', s=350, c=domain_to_color[name], edgecolors='black', linewidths=1.2, zorder=5)
        ax.annotate(short, (dom_coords[j, 0], dom_coords[j, 1]), fontsize=9, fontweight='bold', xytext=(8, 8), textcoords='offset points', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray'))

    ax.set_title(title, fontsize=15, fontweight='bold', pad=15)
    ax.legend(fontsize=9, loc='upper right', ncol=2, framealpha=0.9)
    ax.grid(alpha=0.15)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {save_path}')

def plot_suffix_analysis(categories, baseline_results):
    suffixes = ['station', 'company', 'supplier', 'service', 'center', 'contractor']
    counts = []
    mean_scores = []
    for suffix in suffixes:
        matching = [cat for cat in categories if cat.endswith(suffix)]
        counts.append(len(matching))
        if matching:
            scores = [baseline_results[cat]['score'] for cat in matching if cat in baseline_results]
            mean_scores.append(np.mean(scores) if scores else 0.0)
        else:
            mean_scores.append(0.0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6', '#1abc9c']

    bars1 = ax1.bar(suffixes, counts, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    for bar, c in zip(bars1, counts):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(c), ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax1.set_title('Category Count by Suffix', fontsize=14, fontweight='bold', pad=12)

    bars2 = ax2.bar(suffixes, mean_scores, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    for bar, s in zip(bars2, mean_scores):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{s:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax2.set_title('Mean Baseline Score by Suffix', fontsize=14, fontweight='bold', pad=12)
    ax2.set_ylim(0, 1.0)

    plt.suptitle('Suffix Poisoning Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('figures/suffix_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved: figures/suffix_analysis.png')


In [12]:
print('=' * 60)
print('GENERATING ALL FIGURES')
print('=' * 60)

print('\n[1/4] Confidence Distribution...')
plot_confidence_distribution(baseline_results, lookup)

print('\n[2/4] Per-Domain Accuracy...')
plot_per_domain_accuracy(lookup)

print('\n[3/4] t-SNE Fine-Tuned Embedding Space...')
# Plot using trained model on categories that have valid ground truth
ft_model_for_tsne = SentenceTransformer('./model')
valid_cats = [c for c, gt in ground_truth.items() if gt is not None]
plot_tsne(valid_cats, DOMAINS, ft_model_for_tsne, 't-SNE: Fine-Tuned Embedding Space', 'figures/tsne_finetuned.png')

print('\n[4/4] Suffix Poisoning Analysis...')
# Uses the original baseline_results and CATEGORIES from Phase 1
plot_suffix_analysis(CATEGORIES, baseline_results)

print('\n' + '=' * 60)
print('ALL FIGURES SAVED TO figures/')
print('=' * 60)


GENERATING ALL FIGURES

[1/4] Confidence Distribution...
Saved: figures/confidence_dist.png

[2/4] Per-Domain Accuracy...
Saved: figures/per_domain_accuracy.png

[3/4] t-SNE Fine-Tuned Embedding Space...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Encoding 2023 categories...
  Running t-SNE (this may take a minute)...


/usr/local/lib/python3.13/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


  Saved: figures/tsne_finetuned.png

[4/4] Suffix Poisoning Analysis...
Saved: figures/suffix_analysis.png

ALL FIGURES SAVED TO figures/


In [16]:
import shutil
import os

# Zips only the main model, figures, and outputs (ignoring the heavy backups)
!zip -r backup_fast.zip model/ figures/ outputs/ -x "model/checkpoints/*"

# Instantly copy it to your Google Drive
shutil.copy("backup_fast.zip", "/content/drive/MyDrive/TaxoSplitter_Final_Model.zip")

print("Fast backup complete! Check your Google Drive!")


  adding: model/ (stored 0%)
  adding: model/sentence_bert_config.json (deflated 43%)
  adding: model/config_sentence_transformers.json (deflated 40%)
  adding: model/2_Normalize/ (stored 0%)
  adding: model/README.md (deflated 74%)
  adding: model/1_Pooling/ (stored 0%)
  adding: model/1_Pooling/config.json (deflated 17%)
  adding: model/modules.json (deflated 64%)
  adding: model/model.safetensors (deflated 14%)
  adding: model/config.json (deflated 52%)
  adding: model/tokenizer.json (deflated 71%)
  adding: model/tokenizer_config.json (deflated 45%)
  adding: figures/ (stored 0%)
  adding: figures/suffix_analysis.png (deflated 34%)
  adding: figures/tsne_finetuned.png (deflated 11%)
  adding: figures/per_domain_accuracy.png (deflated 21%)
  adding: figures/confidence_dist.png (deflated 24%)
  adding: outputs/ (stored 0%)
  adding: outputs/final_mapping.json (deflated 91%)
  adding: outputs/baseline_results.json (deflated 89%)
Fast backup complete! Check your Google Drive!
